# DoubleB Particle Transformer Notebook
Full pipeline: ROOT loading → preprocessing → Transformer → training → ROC

In [4]:
import os
import glob
import numpy as np
import awkward as ak
import uproot
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import roc_curve, auc
import matplotlib.pyplot as plt

In [5]:
train_directory = "~/Desktop/DoubleB/train_data/"
test_directory = "~/Desktop/DoubleB/test_data/"

# === Jet features ===
jet_features = [
    'fj_tau1','fj_tau2','fj_tau3','fj_tau21','fj_tau32',
    'fj_nSV','fj_jetNTracks',
    'fj_sdsj1_pt','fj_sdsj1_csv','fj_sdsj2_pt','fj_sdsj2_csv',
    'fj_z_ratio'
]

# === PF candidate features ===
pf_features = [
    "pfcand_ptrel",
    "pfcand_etarel",
    "pfcand_phirel",
    "pfcand_deltaR",
    "pfcand_charge"
]

spectators = ['fj_sdmass','fj_pt']

In [10]:
import os
import glob
import awkward as ak
import uproot

def safe_extract(batch, wanted_features):
    """
    Extract only features that exist.
    Missing ones are filled with zeros.
    """
    out = {}

    for feat in wanted_features:
        if feat in batch.fields:
            out[feat] = batch[feat]
        else:
            # create zero placeholder with correct length
            out[feat] = ak.zeros_like(batch[batch.fields[0]])

    return ak.zip(out)


def load_data(directory, chunk_size=2000, max_files=None):

    files = glob.glob(os.path.expanduser(directory) + "*.root")

    if max_files is not None:
        files = files[:max_files]

    jet_arrays = []
    pf_arrays = []
    labels = []

    for f in files:
        print(f"\nReading: {f}")

        try:
            root_file = uproot.open(f)
            tree = root_file.get("deepntuplizer/tree")

            if tree is None:
                print("⚠️ Tree missing — skipping file")
                continue

        except Exception as e:
            print("❌ Cannot open file:", e)
            continue

        try:
            for batch in tree.iterate(
                library="ak",
                step_size=chunk_size
            ):

                # skip empty chunks
                if len(batch) == 0:
                    continue

                # apply physics cuts only if variables exist
                if "fj_sdmass" in batch.fields and "fj_pt" in batch.fields:
                    mask = (
                        (batch["fj_sdmass"] > 40)
                        & (batch["fj_sdmass"] < 200)
                        & (batch["fj_pt"] > 350)
                        & (batch["fj_pt"] < 2000)
                    )
                    batch = batch[mask]

                if len(batch) == 0:
                    continue

                jets = safe_extract(batch, jet_features)
                pfs = safe_extract(batch, pf_features)
                # === Build CMS physics labels ===

                label_qcd = batch["fj_isQCD"] * batch["sample_isQCD"]
                label_hbb = batch["fj_isH"] * batch["fj_isBB"]

                 # keep only clean signal/background
                 mask_valid = (label_qcd == 1) | (label_hbb == 1)
                 batch = batch[mask_valid]

                 # binary label: H→bb = 1, QCD = 0
                y = ak.where(label_hbb[mask_valid] == 1, 1, 0)

               
               
               
                jet_arrays.append(jets)
                pf_arrays.append(pfs)
                labels.append(y)

        except Exception as e:
            print("⚠️ Chunk read error — salvaging what we got:", e)
            continue

    if len(jet_arrays) == 0:
        raise RuntimeError("No usable data found.")

    jets = ak.concatenate(jet_arrays)
    pfs = ak.concatenate(pf_arrays)
    y = ak.concatenate(labels)

    print("\n✅ Recovered events:", len(jets))
    return jets, pfs, y


In [11]:
class ParticleDataset(Dataset):
    def __init__(self, jets, pfs, labels, max_particles=50):

        self.jets = torch.tensor(ak.to_numpy(jets), dtype=torch.float32)
        self.labels = torch.tensor(labels, dtype=torch.float32)

        # pad / clip PF candidates
        padded = ak.pad_none(pfs, max_particles, clip=True)
        padded = ak.fill_none(padded, 0)
        self.pfs = torch.tensor(ak.to_numpy(padded), dtype=torch.float32)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.pfs[idx], self.jets[idx], self.labels[idx]

In [12]:
class ParticleTransformer(nn.Module):
    def __init__(self, pf_dim, jet_dim, d_model=64, nhead=4, num_layers=3):
        super().__init__()

        self.embed = nn.Linear(pf_dim, d_model)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            batch_first=True
        )

        self.encoder = nn.TransformerEncoder(
            encoder_layer,
            num_layers=num_layers
        )

        self.classifier = nn.Sequential(
            nn.Linear(d_model + jet_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 1),
            nn.Sigmoid()
        )

    def forward(self, pfs, jets):

        x = self.embed(pfs)
        x = self.encoder(x)
        x = x.mean(dim=1)

        x = torch.cat([x, jets], dim=1)
        return self.classifier(x).squeeze()

In [13]:
# === Load data ===
train_jets, train_pfs, y_train = load_data(train_directory)
test_jets, test_pfs, y_test = load_data(test_directory)

train_dataset = ParticleDataset(train_jets, train_pfs, y_train)
test_dataset = ParticleDataset(test_jets, test_pfs, y_test)

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=256)

# === Model ===
model = ParticleTransformer(
    pf_dim=train_pfs.type.content.length,
    jet_dim=len(jet_features)
)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.BCELoss()

def train_epoch(model, loader):
    model.train()
    total_loss = 0

    for pfs, jets, y in loader:
        optimizer.zero_grad()

        preds = model(pfs, jets)
        loss = criterion(preds, y)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)

for epoch in range(10):
    loss = train_epoch(model, train_loader)
    print(f"Epoch {epoch+1}: Loss={loss:.4f}")


Reading: /Users/apple/Desktop/DoubleB/train_data/ntuple_merged_76.root

Reading: /Users/apple/Desktop/DoubleB/train_data/ntuple_merged_21.root

Reading: /Users/apple/Desktop/DoubleB/train_data/ntuple_merged_37.root
❌ Cannot open file: expected Chunk of length 139,
received 0 bytes from FSSpecSource
for file path /Users/apple/Desktop/DoubleB/train_data/ntuple_merged_37.root

Reading: /Users/apple/Desktop/DoubleB/train_data/ntuple_merged_60.root

Reading: /Users/apple/Desktop/DoubleB/train_data/ntuple_merged_17.root

Reading: /Users/apple/Desktop/DoubleB/train_data/ntuple_merged_40.root

Reading: /Users/apple/Desktop/DoubleB/train_data/ntuple_merged_56.root
⚠️ Chunk read error — salvaging what we got: Error -3 while decompressing data: incorrect data check

Reading: /Users/apple/Desktop/DoubleB/train_data/ntuple_merged_83.root

Reading: /Users/apple/Desktop/DoubleB/train_data/ntuple_merged_82.root

Reading: /Users/apple/Desktop/DoubleB/train_data/ntuple_merged_57.root

Reading: /Users/a

TypeError: can't convert np.ndarray of type numpy.void. The only supported types are: float64, float32, float16, complex64, complex128, int64, int32, int16, int8, uint8, and bool.

In [ ]:
model.eval()

scores = []
labels = []

with torch.no_grad():
    for pfs, jets, y in test_loader:
        s = model(pfs, jets)
        scores.extend(s.numpy())
        labels.extend(y.numpy())

scores = np.array(scores)
labels = np.array(labels)

fpr, tpr, _ = roc_curve(labels, scores)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(6,6))
plt.plot(tpr, 1-fpr, label=f"Particle Transformer (AUC={roc_auc:.3f})")
plt.xlabel("Signal efficiency")
plt.ylabel("Background rejection")
plt.legend()
plt.grid(True)
plt.show()